In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
#!ls "/content/drive/MyDrive/Colab Notebooks"


In [ ]:
#!ls

In [ ]:
# Load forensic_explanation notebook
#%run "/content/drive/MyDrive/Colab Notebooks/forensic_explanation.ipynb"

In [ ]:
#print(format_forensic_report)

In [ ]:
#!pip install gradio pandas plotly

In [ ]:
# # ==================================
# # WEB DASHBOARD
# # ==================================

# import gradio as gr
# import pandas as pd
# import numpy as np
# import joblib
# import plotly.graph_objects as go
# from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
# from google.colab import drive


# # ==================================
# # LOAD MODELS
# # ==================================

# drive.mount('/content/drive')

# BASE_PATH="/content/drive/MyDrive/Colab Notebooks"

# model_package = joblib.load(f"{BASE_PATH}/windows_threat_detection_model.pkl")

# model = model_package["model"]
# features = model_package["features"]

# print("Windows model loaded")


# # ==================================
# # FORENSIC EXPLANATION ENGINE
# # ==================================

# def generate_forensic_explanation(row):

#     explanation=[]

#     if row["Event_ID"] == 4625:
#         explanation.append("Repeated failed login attempt detected")

#     if row["Event_ID"] == 4672:
#         explanation.append("Privileged account activity observed")

#     if row["Event_ID"] == 5379:
#         explanation.append("Credential manager access detected")

#     if row["event_freq"] > 20:
#         explanation.append("High event frequency anomaly")

#     if row["burst_count"] > 10:
#         explanation.append("Burst activity detected")

#     if row["night_activity"] == 1:
#         explanation.append("Suspicious night time activity")

#     if len(explanation)==0:
#         explanation.append("No strong anomaly indicators")

#     return " | ".join(explanation)


# # ==================================
# # THREAT GAUGE
# # ==================================

# def create_gauge(score):

#     fig = go.Figure(go.Indicator(

#         mode="gauge+number",
#         value=score,

#         title={'text':"Threat Level"},

#         gauge={

#             'axis':{'range':[0,100]},

#             'bar':{'color':"red"},

#             'steps':[

#                 {'range':[0,30],'color':"green"},
#                 {'range':[30,70],'color':"yellow"},
#                 {'range':[70,100],'color':"red"}

#             ]

#         }

#     ))

#     return fig


# # ==================================
# # FEATURE ENGINEERING
# # ==================================

# def feature_engineering(df):

#     df=df.copy()

#     df.columns = df.columns.str.strip().str.replace(" ", "_").str.lower()

#     df["date_and_time"]=pd.to_datetime(df["date_and_time"],errors="coerce")

#     df=df.sort_values("date_and_time")

#     df["hour"]=df["date_and_time"].dt.hour

#     df["hour_sin"]=np.sin(2*np.pi*df["hour"]/24)
#     df["hour_cos"]=np.cos(2*np.pi*df["hour"]/24)

#     df["night_activity"]=((df["hour"]<6)|(df["hour"]>22)).astype(int)

#     df["event_freq"]=df.groupby("event_id").cumcount()

#     df["burst_count"]=(

#         df.groupby("event_id")
#         .rolling(20)["event_id"]
#         .count()
#         .reset_index(level=0,drop=True)

#     )

#     df["burst_count"]=df["burst_count"].fillna(0)

#     privileged=[4672,4673,4674]

#     df["is_privileged_event"]=df["event_id"].apply(
#         lambda x:1 if x in privileged else 0
#     )

#     df["event_rarity"]=1/(df["event_freq"]+1)

#     df["credential_attack_sequence"]=(

#         (df["event_id"].shift(2)==4624) &
#         (df["event_id"].shift(1)==4672) &
#         (df["event_id"]==5379)

#     ).astype(int)

#     return df


# # ==================================
# # MAIN PIPELINE
# # ==================================

# def process_windows_logs(file):

#     try:

#         df=pd.read_csv(file.name)

#         df=feature_engineering(df)

#         X=df[features]

#         preds=model.predict(X)

#         df["Verdict"]=preds

#         label_map={
#             0:"NORMAL",
#             1:"MEDIUM",
#             2:"HIGH_RISK",
#             3:"CRITICAL"
#         }

#         df["Verdict"]=df["Verdict"].map(label_map)

#         df["Forensic_Explanation"]=df.apply(
#             generate_forensic_explanation,
#             axis=1
#         )


#         # ----------------------------

#         critical=(df["Verdict"]=="CRITICAL").sum()
#         high=(df["Verdict"]=="HIGH_RISK").sum()
#         medium=(df["Verdict"]=="MEDIUM").sum()

#         severity=(critical+high)/len(df)*100

#         gauge=create_gauge(severity)


#         # ----------------------------
#         # REPORT
#         # ----------------------------

#         report=f"""
# WINDOWS FORENSIC ANALYSIS REPORT

# Records analyzed: {len(df)}

# Medium Risk: {medium}
# High Risk: {high}
# Critical: {critical}

# -------------------------

# TOP INCIDENTS
# """

#         top=df[df["Verdict"]!="NORMAL"].head(5)

#         for _,row in top.iterrows():

#             report+=f"""

# Event_ID: {row['event_id']}
# Verdict: {row['Verdict']}
# Explanation: {row['Forensic_Explanation']}

# """


#         # ----------------------------
#         # TABLES
#         # ----------------------------

#         table=df[[
#             "date_and_time",
#             "event_id",
#             "source",
#             "Verdict",
#             "Forensic_Explanation"
#         ]]

#         table.columns=[
#             "Timestamp",
#             "Event ID",
#             "Source",
#             "Verdict",
#             "Explanation"
#         ]

#         return table, df, report, gauge

#     except Exception as e:

#         return pd.DataFrame(), pd.DataFrame(), str(e), create_gauge(0)


# # ==================================
# # DASHBOARD UI
# # ==================================

# with gr.Blocks(theme=gr.themes.Soft()) as demo:

#     gr.Markdown("# 🛡️ AI WINDOWS FORENSIC COMMAND CENTER")

#     with gr.Row():

#         upload_button=gr.UploadButton(
#             "Upload Windows Logs CSV",
#             file_types=[".csv"],
#             file_count="single"
#         )

#     with gr.Tabs():

#         with gr.Tab("📡 Event Verdicts"):
#             verdict_table=gr.Dataframe()

#         with gr.Tab("📄 Full Logs"):
#             full_table=gr.Dataframe()

#         with gr.Tab("🔍 AI Forensic Report"):
#             status_text=gr.Textbox(lines=25)

#         with gr.Tab("⚠️ Threat Severity"):
#             gauge_plot=gr.Plot()


#     upload_button.upload(

#         fn=process_windows_logs,

#         inputs=upload_button,

#         outputs=[verdict_table,full_table,status_text,gauge_plot]

#     )


# demo.launch(share=True)

In [ ]:

!ls "/content/drive/MyDrive"
print("Colab Notebooks folder")
!ls "/content/drive/MyDrive/Colab Notebooks"
print("Mini_Project folder")
!ls "/content/drive/MyDrive/Mini_Project"

 061_SIH-HackerTroupe_Feasibility_Viability.gslides
 ai_based_log_investigation.gdoc
 ai_log.drawio
 AI_Log.gslides
 ai_log_investigation.gdoc
 AI_Log_Investigation.gslides
 Automated_Data_Breach_and_Dark_Web_Exposure_Monitor.gdoc
'Colab Notebooks'
 CyberForensics_BinaryBrains.gdoc
 CyberStorm_Divas_HackersGambit2025_Writeup.gdoc
 CyberStorm_Divas_HackersGambit2025_Writeup.pdf
 df_refined.csv
 Foundational_Quantum_Algorithms_Qiskit_Colab.ipynb
 Hackers_gambit2025_writeups.gdoc
'Lesson plan.gslides'
 Literature_Review.gdoc
 microsoft_otp_login_vuln.gdoc
 mini_project.drawio
 N210340.jpg
 PSID:25184.gdoc
 qml_malware.gdoc
 Untitled
'Untitled document (1).gdoc'
'Untitled document (2).gdoc'
'Untitled document.gdoc'
'Untitled presentation (1).gslides'
'Untitled presentation.gslides'
Colab Notebooks folder
 10685642.zip
 aplication_ai_mlp.ipynb
 app_ai_scaler.pkl
 app_ensemble_threshold.npy
'application_dataset (1).csv.gz'
 application_dataset.csv.gz
 application_gatekeeper.ipynb
 applicatio

In [ ]:
import gradio as gr
import pandas as pd
import json
import plotly.graph_objects as go
import subprocess
import os
from google.colab import drive

drive.mount('/content/drive')

BASE_PATH="/content/drive/MyDrive/Colab Notebooks/"
def run_pipeline(file_obj):

    import pandas as pd
    import json
    import traceback
    from IPython import get_ipython

    try:
        dataset_path = BASE_PATH + "uploaded_logs.csv"

        # =============================
        # Save uploaded file
        # =============================

        if file_obj is None:
            raise ValueError("No file uploaded")


        df = pd.read_csv(file_obj.name)
        df.to_csv(dataset_path, index=False)

        print("Uploaded dataset saved:", dataset_path)
        print("Rows:", len(df))
        print("Columns:", list(df.columns))

        # =============================
        # RUN PIPELINE
        # =============================

        # print("Stage 1: Feature Engineering")
        # get_ipython().run_line_magic('run', f'"{BASE_PATH}windows_logs.ipynb"')

        print("Stage 1: Feature_Engineering & AI Detection")
        get_ipython().run_line_magic('run', f'"{BASE_PATH}windows_ai.ipynb"')

        print("Stage 2: Gatekeeper Filtering")
        get_ipython().run_line_magic('run', f'"{BASE_PATH}windows_gatekeeper.ipynb"')

        print("Stage 3: Attack Correlation")
        get_ipython().run_line_magic('run', f'"{BASE_PATH}windows_flink.ipynb"')


        print("Stage 4:MLP Anomaly detection")
        get_ipython().run_line_magic('run', f'"{BASE_PATH}windows_mlp_inference.ipynb"')

        print("Stage 5: Forensic Analysis")
        get_ipython().run_line_magic('run', f'"{BASE_PATH}forensic_explanation.ipynb"')

        # =============================
        # LOAD RESULTS
        # =============================

        alerts = pd.read_csv(BASE_PATH + "windows_attack_patterns.csv")

        mlp_alerts = pd.read_csv(BASE_PATH + "windows_mlp_alerts.csv")
        with open(BASE_PATH + "forensic_reports.json") as f:
            reports = json.load(f)

        reports = pd.DataFrame(reports)
        total_logs = len(pd.read_csv(BASE_PATH + "uploaded_logs.csv"))

        threat_fig = generate_threat_meter(alerts, mlp_alerts, total_logs)

        return alerts, mlp_alerts, reports, threat_fig
    except Exception as e:
        error =str(e)
        trace = traceback.format_exc()
        print("Pipeline Failed")
        print(trace)
        error_df= pd.DataFrame({"Error":[error]})
        trace_df= pd.DataFrame({"Trace":[trace]})
        return error_df, error_df,error_df

def generate_threat_meter(alerts, mlp_alerts, total_logs):

    attack_count = len(alerts)
    anomaly_count = len(mlp_alerts)

    attack_rate = attack_count / total_logs
    anomaly_rate = anomaly_count / total_logs

    anomaly_rate = min(anomaly_rate, 0.15)

    attack_weight = 0.8
    anomaly_weight = 0.2

    threat_score = (
        attack_rate * attack_weight +
        anomaly_rate * anomaly_weight
    ) * 100

    threat_score = round(threat_score, 2)

    fig = go.Figure(go.Indicator(
        mode="gauge+number",
        value=threat_score,
        title={"text": "SOC Threat Level"},
        gauge={
            "axis": {"range":[0,100]},
            "bar":{"color":"red"},
            "steps":[
                {"range":[0,20],"color":"green"},
                {"range":[20,40],"color":"yellow"},
                {"range":[40,70],"color":"orange"},
                {"range":[70,100],"color":"red"}
            ]
        }
    ))

    return fig
# ======================================
# UI
# ======================================
css = """
body {
background: linear-gradient(135deg,#020617,#020617,#020617,#0f172a);
}

.gradio-container {
max-width:1400px !important;
margin:auto;
}

.panel{
background:rgba(15,23,42,0.75);
border-radius:18px;
padding:25px;
border:1px solid rgba(56,189,248,0.4);
box-shadow:0 0 25px rgba(56,189,248,0.15);
}

h1{
font-size:36px !important;
}

"""

# ======================================
# DASHBOARD UI
# ======================================

with gr.Blocks(css=css, theme=gr.themes.Soft()) as demo:

    gr.HTML(
        "<h1 style='text-align:center;color:#38bdf8'>🛡️ AI WINDOWS FORENSIC COMMAND CENTER</h1>"
    )

    with gr.Row():

        module = gr.Radio(
            ["WINDOWS LOGS"],
            value="WINDOWS LOGS",
            label="Module"
        )

        file_input = gr.UploadButton(
            "Upload Windows Log CSV",
            file_types=[".csv"]
        )

    with gr.Tabs():

        # ==============================
        # ATTACK ALERTS
        # ==============================

        with gr.Tab("🚨 Detected Attacks"):

            alerts_table = gr.Dataframe(
                label="MITRE Attack Detection",
                interactive=False
            )
        with gr.Tab("🧠 Deep Learning Alerts"):

            mlp_table = gr.Dataframe(
                label="MLP Anomaly Detection",
                interactive=False
            )
        # ==============================
        # FORENSIC REPORTS
        # ==============================

        with gr.Tab("🔍 Forensic Reports"):

            forensic_table = gr.Dataframe(
                label="Cryptographically Signed Forensic Evidence",
                interactive=False
            )
        with gr.Tab("🛡️ Threat Meter"):

            threat_gauge = gr.Plot(
                label="Real-Time SOC Threat Indicator"
            )
        # ==============================
        # SYSTEM INFO PANEL
        # ==============================

        with gr.Tab("📊 Investigation Info"):

            info_box = gr.Markdown("""
                ## 🔎 SOC Investigation Overview

                ### Pipeline Architecture

                | Stage | Component | Purpose |
                |------|------|------|
                | 1 | Feature Engineering | Extract temporal and behavioral log features |
                | 2 | AI Threat Detection | RandomForest behavioral threat scoring |
                | 3 | Gatekeeper Filtering | Separate normal vs suspicious logs |
                | 4 | MITRE ATT&CK Engine | Correlate logs with attack patterns |
                | 5 | Deep Learning Model | Detect unknown behavioral anomalies |
                | 6 | Explainable AI | SHAP model explanations |
                | 7 | Forensic Engine | Evidence generation and reporting |
                | 8 | Cryptographic Signing | Tamper-proof forensic evidence |

                ---

                ### Security Capabilities

                ✔ Behavioral threat detection
                ✔ MITRE ATT&CK mapping
                ✔ Deep learning anomaly detection
                ✔ Explainable AI forensic reasoning
                ✔ Cryptographically signed digital evidence
                ✔ SOC visual investigation dashboard

                ---

                ### Evidence Integrity

                Each forensic report contains:

                • SHA256 evidence hash
                • ECDSA digital signature
                • Model version traceability
                • Timestamped forensic artifact

                These guarantees ensure **tamper-proof incident documentation** suitable for forensic investigation.
                """)

    file_input.upload(
        run_pipeline,
        inputs=file_input,
        outputs=[alerts_table, mlp_table, forensic_table, threat_gauge]
    )


demo.launch(share=True, debug=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/tmp/ipykernel_203/4073946216.py:153: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(css=css, theme=gr.themes.Soft()) as demo:
/tmp/ipykernel_203/4073946216.py:153: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=css, theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://2cc62ffe64db80b164.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# import gradio as gr
# import pandas as pd
# import plotly.express as px
# import plotly.graph_objects as go


# # ============================
# # MITRE ATTACK MAP
# # ============================

# MITRE_ATTACKS = {

# 4625:("Brute Force Login","T1110"),
# 5379:("Credential Dump","T1003"),
# 4672:("Privilege Escalation","T1068"),
# 4798:("Account Enumeration","T1087"),
# 4799:("Account Enumeration","T1087"),
# 4616:("Time Manipulation","T1070"),
# 4648:("Token Impersonation","T1134")

# }


# # ============================
# # FORENSIC EXPLANATION
# # ============================
# def explain_event(event):

#     explanations=[]

#     if event==4625:
#         explanations.append("Multiple authentication failures detected. This pattern commonly appears during brute force or password spray attempts.")

#     if event==4624:
#         explanations.append("Successful login event recorded. If preceded by failures it may indicate brute force success.")

#     if event==4672:
#         explanations.append("Administrative privileges were assigned to a user session. Attackers often escalate privileges after initial access.")

#     if event==5379:
#         explanations.append("Credential manager accessed. This may indicate credential dumping or password extraction attempts.")

#     if event in [4798,4799]:
#         explanations.append("Account discovery activity detected. Attackers enumerate users before lateral movement.")

#     if event==4648:
#         explanations.append("Explicit credential logon detected. This can indicate lateral movement or impersonation.")

#     if event==4616:
#         explanations.append("System time modification detected. Attackers sometimes modify time to evade forensic analysis.")

#     if event==4697:
#         explanations.append("New service installation detected. Malware often installs persistent services.")

#     if event==4698:
#         explanations.append("Scheduled task created. Attackers use scheduled tasks for persistence.")

#     if event==1102:
#         explanations.append("Security logs cleared. This is a strong defense-evasion indicator.")

#     if len(explanations)==0:
#         explanations.append("No suspicious indicators detected. Event appears consistent with normal system activity.")

#     return " | ".join(explanations)

# # ============================
# # RISK ENGINE
# # ============================

# def calculate_risk(df):

#     df["risk_score"]=0

#     df.loc[df["event_id"]==4625,"risk_score"]=1
#     df.loc[df["event_id"]==4672,"risk_score"]=2
#     df.loc[df["event_id"]==5379,"risk_score"]=3

#     return df


# # ============================
# # ATTACK PATTERN DETECTION
# # ============================

# def detect_attacks(df):

#     events=df["event_id"].tolist()
#     times=df["date_and_time"].tolist()

#     detected=[]

#     for i in range(len(events)-4):

#         # =========================
#         # BRUTE FORCE LOGIN
#         # =========================
#         if events[i]==4625:

#             count=1
#             j=i+1

#             while j<len(events) and (times[j]-times[i]).total_seconds()<120:

#                 if events[j]==4625:
#                     count+=1

#                 j+=1

#             if count>=8:

#                 detected.append({

#                 "attack":"Brute Force Login",
#                 "mitre":"T1110",
#                 "start":times[i],
#                 "end":times[j-1]

#                 })


#         # =========================
#         # PASSWORD SPRAY
#         # =========================
#         if events[i]==4625 and events[i+1]==4624:

#             detected.append({

#             "attack":"Password Spray Attempt",
#             "mitre":"T1110.003",
#             "start":times[i],
#             "end":times[i+1]

#             })


#         # =========================
#         # PRIVILEGE ESCALATION
#         # =========================
#         if events[i]==4624 and events[i+1]==4672:

#             detected.append({

#             "attack":"Privilege Escalation",
#             "mitre":"T1068",
#             "start":times[i],
#             "end":times[i+1]

#             })


#         # =========================
#         # CREDENTIAL DUMP
#         # =========================
#         if events[i]==4624 and events[i+1]==4672 and events[i+2]==5379:

#             detected.append({

#             "attack":"Credential Dump",
#             "mitre":"T1003",
#             "start":times[i],
#             "end":times[i+2]

#             })


#         # =========================
#         # ACCOUNT ENUMERATION
#         # =========================
#         if events[i] in [4798,4799]:

#             count=1
#             j=i+1

#             while j<len(events) and (times[j]-times[i]).total_seconds()<180:

#                 if events[j] in [4798,4799]:
#                     count+=1

#                 j+=1

#             if count>=8:

#                 detected.append({

#                 "attack":"Account Enumeration",
#                 "mitre":"T1087",
#                 "start":times[i],
#                 "end":times[j-1]

#                 })


#         # =========================
#         # LATERAL MOVEMENT
#         # =========================
#         if events[i]==4624 and events[i+1]==4648:

#             detected.append({

#             "attack":"Possible Lateral Movement",
#             "mitre":"T1021",
#             "start":times[i],
#             "end":times[i+1]

#             })


#         # =========================
#         # TOKEN IMPERSONATION
#         # =========================
#         if events[i]==4648:

#             detected.append({

#             "attack":"Token Impersonation",
#             "mitre":"T1134",
#             "start":times[i],
#             "end":times[i]

#             })


#         # =========================
#         # SYSTEM TIME TAMPERING
#         # =========================
#         if events[i]==4616:

#             detected.append({

#             "attack":"System Time Manipulation",
#             "mitre":"T1070",
#             "start":times[i],
#             "end":times[i]

#             })


#         # =========================
#         # SERVICE INSTALLATION
#         # =========================
#         if events[i]==4697:

#             detected.append({

#             "attack":"Suspicious Service Installation",
#             "mitre":"T1543",
#             "start":times[i],
#             "end":times[i]

#             })


#         # =========================
#         # PERSISTENCE VIA TASK
#         # =========================
#         if events[i]==4698:

#             detected.append({

#             "attack":"Scheduled Task Persistence",
#             "mitre":"T1053",
#             "start":times[i],
#             "end":times[i]

#             })


#         # =========================
#         # DEFENSE EVASION
#         # =========================
#         if events[i]==1102:

#             detected.append({

#             "attack":"Security Log Cleared",
#             "mitre":"T1070",
#             "start":times[i],
#             "end":times[i]

#             })


#     return pd.DataFrame(detected)

# # ============================
# # THREAT GAUGE
# # ============================

# def threat_gauge(score):

#     fig=go.Figure(go.Indicator(

#     mode="gauge+number",

#     value=score,

#     title={'text':"Threat Level"},

#     gauge={

#     'axis':{'range':[0,100]},

#     'steps':[

#     {'range':[0,30],'color':"green"},
#     {'range':[30,70],'color':"yellow"},
#     {'range':[70,100],'color':"red"}

#     ]

#     }

#     ))

#     return fig


# # ============================
# # MAIN PIPELINE
# # ============================

# def run_pipeline(file):

#     df=pd.read_csv(file.name)

#     df.columns=df.columns.str.lower().str.replace(" ","_")

#     df["date_and_time"]=pd.to_datetime(df["date_and_time"])

#     df=df.sort_values("date_and_time")

#     df=calculate_risk(df)

#     df["explanation"]=df["event_id"].apply(explain_event)

#     attacks=detect_attacks(df)

#     high=(df["risk_score"]>=2).sum()

#     severity=(high/len(df))*100

#     gauge=threat_gauge(severity)

#     # ============================
#     # ATTACK TIMELINE
#     # ============================

#     timeline=px.histogram(

#     df,
#     x="date_and_time",
#     color="risk_score",
#     title="Security Events Timeline"

#     )

#     # ============================
#     # TOP SUSPICIOUS EVENTS
#     # ============================

#     top_events=px.bar(

#     df["event_id"].value_counts().head(10),

#     title="Most Frequent Event IDs"

#     )

#     display=df[[
#     "date_and_time",
#     "event_id",
#     "source",
#     "risk_score",
#     "explanation"
#     ]]

#     return display,attacks,gauge,timeline,top_events


# # ============================
# # DASHBOARD UI
# # ============================

# with gr.Blocks() as demo:

#     gr.Markdown("# 🛡️ Windows AI SOC Dashboard")

#     upload=gr.UploadButton(
#     "Upload Windows Logs CSV",
#     file_types=[".csv"]
#     )

#     with gr.Tabs():

#         with gr.Tab("Event Analysis"):
#             table=gr.Dataframe()

#         with gr.Tab("Detected Attacks"):
#             attacks=gr.Dataframe()

#         with gr.Tab("Threat Severity"):
#             gauge=gr.Plot()

#         with gr.Tab("Attack Timeline"):
#             timeline=gr.Plot()

#         with gr.Tab("Top Events"):
#             top_events=gr.Plot()

#     upload.upload(
#     run_pipeline,
#     inputs=upload,
#     outputs=[table,attacks,gauge,timeline,top_events]
#     )

# demo.launch(share=True)

In [ ]:
# import gradio as gr
# import pandas as pd
# import numpy as np
# import joblib
# import plotly.graph_objects as go
# from google.colab import drive

# drive.mount('/content/drive')

# BASE_PATH="/content/drive/MyDrive/Colab Notebooks"

# model=joblib.load(f"{BASE_PATH}/windows_ai_model.pkl")
# scaler=joblib.load(f"{BASE_PATH}/windows_scaler.pkl")
# encoders=joblib.load(f"{BASE_PATH}/windows_encoders.pkl")


# # ======================================
# # FEATURE LIST
# # ======================================

# features=[
# "hour_sin",
# "hour_cos",
# "Source",
# "Task_Category",
# "event_freq",
# "event_type_freq",
# "night_activity",
# "burst_count"
# ]


# # ======================================
# # THREAT GAUGE
# # ======================================

# def create_gauge(score):

#     fig=go.Figure(go.Indicator(
#         mode="gauge+number",
#         value=score,
#         title={'text':"Threat Level"},
#         gauge={
#             'axis':{'range':[0,100]},
#             'bar':{'color':"#ef4444"},
#             'steps':[
#                 {'range':[0,30],'color':'#22c55e'},
#                 {'range':[30,70],'color':'#eab308'},
#                 {'range':[70,100],'color':'#ef4444'}
#             ]
#         }
#     ))

#     fig.update_layout(
#         paper_bgcolor="rgba(0,0,0,0)",
#         font_color="white"
#     )

#     return fig


# # ======================================
# # WINDOWS LOG PREPROCESSING
# # ======================================

# def preprocess_windows_logs(df):

#     # Fix column mapping (Event Viewer export)
#     df=df.rename(columns={
#         "Keywords":"Date_and_Time",
#         "Date and Time":"Source",
#         "Source":"Event_ID",
#         "Event ID":"Task_Category",
#         "Task Category":"Message"
#     })

#     # Ensure required columns exist
#     for col in ["Date_and_Time","Source","Event_ID","Task_Category"]:
#         if col not in df.columns:
#             df[col]="Unknown"

#     # Convert datetime
#     df["Date_and_Time"]=pd.to_datetime(
#         df["Date_and_Time"],
#         errors="coerce"
#     )

#     # Convert event id
#     df["Event_ID"]=pd.to_numeric(df["Event_ID"],errors="coerce")

#     # Extract hour
#     df["hour"]=df["Date_and_Time"].dt.hour

#     # Cyclic time encoding
#     df["hour_sin"]=np.sin(2*np.pi*df["hour"]/24)
#     df["hour_cos"]=np.cos(2*np.pi*df["hour"]/24)

#     # Frequency features
#     df["event_freq"]=df.groupby("Event_ID")["Event_ID"].transform("count")
#     df["event_type_freq"]=df.groupby("Task_Category")["Task_Category"].transform("count")

#     # Night activity
#     df["night_activity"]=df["hour"].apply(
#         lambda x:1 if x<6 or x>22 else 0
#     )

#     # Burst detection
#     df["burst_count"]=df.groupby("Event_ID").cumcount()

#     return df


# # ======================================
# # MAIN PROCESSING FUNCTION
# # ======================================

# def process_windows_logs(file_obj):

#     df=pd.read_csv(file_obj.name)

#     print("Logs loaded:",len(df))

#     df=preprocess_windows_logs(df)

#     # Encode categorical features
#     for col,encoder in encoders.items():

#         df[col]=df[col].astype(str)

#         known=set(encoder.classes_)

#         df[col]=df[col].apply(
#             lambda x:x if x in known else encoder.classes_[0]
#         )

#         df[col]=encoder.transform(df[col])

#     # Ensure all model features exist
#     for col in features:
#         if col not in df.columns:
#             df[col]=0

#     X=df[features]

#     X_scaled=scaler.transform(X)

#     # Anomaly score
#     scores=model.decision_function(X_scaled)

#     df["risk_score"]=1-(scores-scores.min())/(scores.max()-scores.min())

#     # Behavioral gatekeeper
#     df["risk_window"]=df["risk_score"].rolling(15).sum()

#     df["gatekeeper_flag"]=df["risk_window"].apply(
#         lambda x:1 if x>=10 else 0
#     )

#     df["Verdict"]=np.where(
#         df["gatekeeper_flag"]==1,
#         "MALICIOUS",
#         "BENIGN"
#     )

#     malicious=(df["gatekeeper_flag"]==1).sum()

#     severity=(malicious/len(df))*100

#     gauge=create_gauge(severity)

#     report=f"""
# WINDOWS FORENSIC REPORT
# ========================

# Logs analyzed : {len(df)}

# Malicious detected : {malicious}

# Threat level : {severity:.2f}%
# """

#     display=df[[
#         "Date_and_Time",
#         "Event_ID",
#         "Task_Category",
#         "risk_score",
#         "Verdict"
#     ]]

#     return display,df,report,gauge


# # ======================================
# # GRADIO DASHBOARD
# # ======================================

# with gr.Blocks() as demo:

#     gr.Markdown("# Windows Cyber Forensic Dashboard")

#     file_input=gr.File(label="Upload Windows Event Logs CSV")

#     verdict_table=gr.Dataframe(label="Suspicious Events")
#     full_table=gr.Dataframe(label="Full Log Analysis")

#     report_box=gr.Textbox(lines=12,label="Investigation Report")

#     gauge_plot=gr.Plot(label="Threat Level")

#     file_input.change(
#         process_windows_logs,
#         inputs=file_input,
#         outputs=[verdict_table,full_table,report_box,gauge_plot]
#     )

# demo.launch(share=True)